# ME 313 · Lab 1.1 — Learn the Convection Coefficient *from data*
**Module 1 companion.**  The notes stress that the convection coefficient **h is not a material property** — it depends on the flow. That is exactly why engineers often *learn* it from data instead of reading a single textbook correlation.

In this lab you generate a (noisy) dataset of Nusselt numbers, fit it two ways, and check what you recovered against the known correlation. **Free Colab, no GPU needed.**

*To make a student fill-in version, blank the lines marked `# TODO`.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split


### 1. Make a dataset
Ground truth is a Dittus–Boelter-style law $Nu = 0.023\,Re^{0.8}Pr^{0.4}$, with 5% measurement noise — as if from experiments or CFD.


In [ ]:
rng = np.random.default_rng(0)
Re = rng.uniform(1e4, 1e5, 400)
Pr = rng.uniform(0.7, 12, 400)
Nu_true = 0.023 * Re**0.8 * Pr**0.4
Nu = Nu_true * (1 + rng.normal(0, 0.05, Re.size))   # add 5% noise
X = np.c_[np.log(Re), np.log(Pr)]      # features in log space
y = np.log(Nu)                          # target in log space
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=1)


### 2. Approach A — linear regression in log space
A power law is linear in logs: $\ln Nu = \ln C + m\ln Re + n\ln Pr$. So linear regression should **recover the exponents**.


In [ ]:
lin = LinearRegression().fit(Xtr, ytr)   # TODO: fit the model
m_Re, n_Pr = lin.coef_
C = np.exp(lin.intercept_)
print(f'learned:  C={C:.4f}, Re-exp={m_Re:.3f}, Pr-exp={n_Pr:.3f}')
print('truth:    C=0.0230, Re-exp=0.800, Pr-exp=0.400')


### 3. Approach B — a small neural network
A neural net needs no assumed form — it just learns the mapping.


In [ ]:
net = make_pipeline(StandardScaler(),
                    MLPRegressor(hidden_layer_sizes=(32,32), max_iter=8000, random_state=0))
net.fit(Xtr, ytr)
models = [('linear', lin), ('neural net', net)]
for name, mdl in models:
    p = mdl.predict(Xte)
    r2 = 1 - np.sum((yte-p)**2)/np.sum((yte-yte.mean())**2)
    print(f'{name:10s} test R^2 = {r2:.4f}')


### 4. Verify — parity plot
Points on the 45° line mean the model reproduces the held-out truth.


In [ ]:
plt.figure(figsize=(5,5))
for (name,mdl),mk in zip(models, ['o','x']):
    plt.scatter(np.exp(yte), np.exp(mdl.predict(Xte)), s=14, marker=mk, alpha=.6, label=name)
lims=[np.exp(yte).min(), np.exp(yte).max()]
plt.plot(lims, lims, 'k--', lw=1)
plt.xlabel('true Nu'); plt.ylabel('predicted Nu'); plt.legend(); plt.title('Lab 1.1 — learned vs. true Nu')
plt.tight_layout(); plt.show()


### 5. Reflect
- The linear fit recovers $\approx(0.8, 0.4)$ — the physics was hiding in the data.
- **Why can't you just look up *h* for “air”?** It depends on $Re$ and $Pr$ (the flow), not just the fluid.
- **Verification habit:** a data-driven *h* is only trustworthy inside the range of data it was trained on. Try predicting at $Re=5\times10^5$ (outside training) and see the error grow.
